In [1]:
import pandas as pd
import numpy as np

def to_num(s):
    return pd.to_numeric(s.astype(str).str.replace(',','.'), errors='coerce')

# Load Gini file - extract both Gini and P80/P20
df = pd.read_csv('/Users/finn/Desktop/TFG/GINI_file.csv', sep='\t', encoding='utf-8-sig')

# Filter to municipality level only
muni = df[
    df['Municipalities'].str.match(r'^\d{5} ', na=False) &
    df['Districts'].isna() &
    df['Sections'].isna()
].copy()

muni['value'] = to_num(muni['Total'])
muni['city'] = muni['Municipalities'].str[6:].str.strip()
muni['cod_ine'] = muni['Municipalities'].str[:5]

# Pivot to wide - one column for Gini, one for P80/P20
wide = muni.pivot_table(
    index=['city','cod_ine','Periodo'],
    columns='Average income indicators',
    values='value',
    aggfunc='first'
).reset_index()

wide.columns = ['city','cod_ine','year','gini','p80p20']
wide = wide.dropna(subset=['gini'])

print(f'Dataset: {len(wide)} rows, {wide.city.nunique()} municipalities')
print(f'Years: {sorted(wide.year.unique())}')
print(f'P80/P20 missing: {wide.p80p20.isna().sum()}')
print(wide.head(5).to_string())

wide.to_csv('/Users/finn/Desktop/TFG/gini_and_p80p20.csv', index=False)
print('Saved!')

Dataset: 59218 rows, 6911 municipalities
Years: [np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
P80/P20 missing: 42
     city cod_ine  year  gini  p80p20
0  Abades   40001  2015  29.3     2.4
1  Abades   40001  2016  27.4     2.4
2  Abades   40001  2017  25.9     2.5
3  Abades   40001  2018  24.4     2.2
4  Abades   40001  2019  23.1     2.2
Saved!
